In [4]:
import os
import pandas as pd
import json
from bs4 import BeautifulSoup
import textstat

In [5]:
# method to read mongo database collection
def read_collection(file_path):

    # Step 1: Try reading as a JSON array
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except json.JSONDecodeError:
        # Step 2: If it's newline-delimited JSON
        with open(file_path, 'r', encoding='utf-8') as f:
            data = [json.loads(line) for line in f]

    # Step 3: Clean MongoDB special fields ($oid, $date, etc.)
    def extract_mongo_specials(obj):
        if isinstance(obj, dict):
            if "$oid" in obj:
                return obj["$oid"]
            if "$date" in obj:
                return obj["$date"]
            return {k: extract_mongo_specials(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [extract_mongo_specials(i) for i in obj]
        return obj

    cleaned_data = [extract_mongo_specials(doc) for doc in data]

    # Step 4: Flatten into DataFrame
    df = pd.json_normalize(cleaned_data)

    return df

[T] Final text stats

In [8]:
docs = [
    "67ce983b285fac3f07bcf5e8", 
    "67c97527285fac3f07bcf587",
    #"67c88f88285fac3f07bcf585", #too complex, drafty
    #"67d43b390b9539e69a3a86b9", #too complex, drafty
   "67d5d8160b9539e69a3a86d8",
    "67d08e87285fac3f07bcf64b",
    "68833fd687cf5fec60dc801d",
    "6879e8fd87cf5fec60dc7fe1",
    "688364bf87cf5fec60dc8026",
    "6877c26e87cf5fec60dc7fda",
     #"6889e57087cf5fec60dc8092", #too complex, drafty
    "688c6add87cf5fec60dc80ef",
    #"68907a6e87cf5fec60dc8107", #GERMAN TEXT => INCREASED SCORE!
    "6896071ee90ffc190a96c285"
]

# read document collection
path_to_doc = os.path.join('input/database', 'document.json')

data = read_collection(path_to_doc)

document_data = data[data["_id"].isin(docs)]

display(document_data)

parsed_text_list = []
word_counts = []
flesch_reading_ease = []
flesch_kincaid_grade = []

final_text_list = []

for idx, row in document_data.iterrows():
    print(row["_id"])
    text = row["text"]
    
    parsed_text = BeautifulSoup(str(text), 'lxml').get_text()
    parsed_text_list.append(parsed_text)
    
    final_text_list.append({"id": row["_id"], "raw": text, "parsed": parsed_text})

    word_counts.append(textstat.lexicon_count(text, removepunct=True))
    
    flesch_reading_ease.append(textstat.flesch_reading_ease(text))    
    flesch_kincaid_grade.append(textstat.flesch_kincaid_grade(text))
    
    #display(text)

display(final_text_list)
final_text_df = pd.DataFrame(data = final_text_list, columns = ["id", "raw", "parsed"])
final_text_df.to_csv(os.path.join('output', 'final', 'final_texts.tsv'), sep="\t")



display(parsed_text_list)    
display(word_counts)
display(flesch_reading_ease)
display(flesch_kincaid_grade)


,_id,cards,created,goal,modified,name,text,user_ids,encoded_doc.$binary.base64,encoded_doc.$binary.subType
1,67c97527285fac3f07bcf587,"{""Ecs05qu-TPe_ExB8Or_bT"":{""mode"":""comment"",""ty...",2025-03-06T10:12:55.000Z,Write your document goal,2025-03-07T21:52:17.000Z,Untitled document,"<h1 class=""ql-align-center""><span class=""annot...","[67a0d6209f5c900ebf5c43ab, 67cb6623285fac3f07b...",DhPokKS9DwCBxqDunAulAQGBxqDunAuoAQOBxYKt7gwAAY...,00
2,67ce983b285fac3f07bcf5e8,"{""1owyGmRTxcq7Y_rSKaC79"":{""id"":""1owyGmRTxcq7Y_...",2025-03-10T07:43:55.000Z,Write your document goal,2025-03-10T08:43:22.000Z,Text by Andrei and Galina,"<h1 class=""ql-align-center""><strong>Health and...","[67ce982f285fac3f07bcf5e7, 67ce9cb4285fac3f07b...",Bp4hl4e1qg4AwbOYpc0GkgGzmKXNBokBAYGzmKXNBpgBAo...,00
3,67d08e87285fac3f07bcf64b,"{""GM02Orml_G34bFo3rNVIp"":{""typing"":[],""state"":...",2025-03-11T19:27:03.000Z,Write your document goal,2025-03-21T11:15:59.000Z,Velikie Testirovschiki,<h3><strong>The Pros and Cons of AI and Its Us...,"[67a0d6209f5c900ebf5c43ab, 67d82dc4edfe680c1b0...",HASc5+T1DwCBr62Y5wZgFIGvrZjnBmEBga+tmOcGAQGBr6...,00
5,67d5d8160b9539e69a3a86d8,"{""AtLIwKR-MP42twebk2MyV"":{""source"":""remote"",""h...",2025-03-15T19:42:14.000Z,It is a seminar paper for a seminar on qualita...,2025-03-31T06:15:01.000Z,HA3,\nInterviewing is one of the central methods i...,"[67d5d8120b9539e69a3a86d7, 67d5bdff0b9539e69a3...",E6IDgJOR6Q8AQd/78ZICAA3BgJOR6Q8M3/vxkgIABMGAk5...,00
6,6877c26e87cf5fec60dc7fda,"{""baXQLp213f0v2zOXugcIO"":{""typing"":[],""mode"":""...",2025-07-16T15:17:02.000Z,Make a detailed plan of our wedding,2025-08-11T09:15:02.000Z,Wedding Planning,<p>This document will outline the plan for our...,"[67a0d6209f5c900ebf5c43ab, 6888bbce87cf5fec60d...",FA/RurDcDwCB1vr4qA0BASEBBWNhcmRzFW5pcGhSV1A1Vn...,00
7,6879e8fd87cf5fec60dc7fe1,"{""5mtT24NvpYG6VGxCCtwa6"":{""id"":""5mtT24NvpYG6VG...",2025-07-18T06:26:05.000Z,refine the discussion with more reflections on...,2025-07-30T11:31:25.000Z,Discussion,<h2>Designing for Emotional Engagement</h2><p>...,"[67a0d6209f5c900ebf5c43ab, 687de5d187cf5fec60d...",EgH276ynDwABAQhkb2N1bWVudBTjDrW9xdsOAIHk35izBA...,00
8,68833fd687cf5fec60dc801d,"{""zJrFbQqnqJS4aAH7SON4E"":{""mode"":""comment"",""id...",2025-07-25T08:27:02.000Z,Provide insight into the matter of AI and it's...,2025-07-29T12:44:51.000Z,An Essay about the impact of AI,"<h1><span class=""annotation comment"" annotatio...","[67a0d6209f5c900ebf5c43ab, 68835b0587cf5fec60d...",DsgBu7u0tA8AwYiwrNYBS4iwrNYBTwGBiLCs1gGjEQHBu7...,00
9,688364bf87cf5fec60dc8026,"{""qqJEAdTTgHiFc3NfCuA6z"":{""type"":""comment"",""re...",2025-07-25T11:04:31.000Z,Website Development Process,2025-08-01T12:25:26.000Z,Phases of Website Development,<h1>The 6 main phases of website development</...,"[67a0d6209f5c900ebf5c43ab, 6888eec887cf5fec60d...",EwOs1aSpDgDBloKd0wXP6QHjqJYrBgGB+ef4zgjYIyaB+e...,00
11,688c6add87cf5fec60dc80ef,"{""RcSGxx7ZOm-9lvxxp2Vu5"":{""type"":""comment"",""ta...",2025-08-01T07:21:01.000Z,Write your document goal,2025-08-27T06:55:02.000Z,Untitled document,<p>Sachbeihilfe DFG</p><h1>Starting Point</h1>...,"[67a0d6209f5c900ebf5c43ab, 6898407be90ffc190a9...",FFSmzpyrDwDB6pWU+w0FzZK1swcBtwSBgp2v3Ag8F4GOpZ...,00
12,6896071ee90ffc190a96c285,"{""kOsaJpu4xvWjvFLq_vQOj"":{""history"":[{""time"":1...",2025-08-08T14:18:06.000Z,Text about our experience as Deustche Telekom ...,2025-08-14T18:21:24.000Z,Scholarship-Essay,<p>The Female@Tech Scholarship sponsored by De...,"[67a0d6209f5c900ebf5c43ab, 689e2181e90ffc190a9...",CBSN0e/SDgCBy5+S8gG0FgOBy5+S8gHAFgSBy5+S8gHYFg...,00


67c97527285fac3f07bcf587
67ce983b285fac3f07bcf5e8
67d08e87285fac3f07bcf64b
67d5d8160b9539e69a3a86d8
6877c26e87cf5fec60dc7fda
6879e8fd87cf5fec60dc7fe1
68833fd687cf5fec60dc801d
688364bf87cf5fec60dc8026
688c6add87cf5fec60dc80ef
6896071ee90ffc190a96c285


[{'id': '67c97527285fac3f07bcf587',
  'raw': '<h1 class="ql-align-center"><span class="annotation comment" annotation-type="comment" annotation-id="olS9Bs3EXpFr5IuZOxEcn" annotation-user="[object Object]" annotation-task="67cb69c8285fac3f07bcf5b2">The Pros and Cons of AI</span></h1><p class="ql-align-center"><br></p><p><span class="annotation comment" annotation-type="comment" annotation-id="XRaeas59fz_KlwuPRZJ5z" annotation-user="[object Object]" annotation-task="67cb69c8285fac3f07bcf5b2">In this essay, we will delve into the various advantages and challenges associated with Artificial Intelligence (AI).</span> Our journey will take us through the transformative potential AI holds for countless industries, along with its implications on society, economy, and ethics. As we navigate through these complexities, we aim to provide a balanced view of AI, illuminating the key issues and breakthroughs. </p><p>First and foremost, AI has the ability to revolutionize industries and enhance human

["The Pros and Cons of AIIn this essay, we will delve into the various advantages and challenges associated with Artificial Intelligence (AI). Our journey will take us through the transformative potential AI holds for countless industries, along with its implications on society, economy, and ethics. As we navigate through these complexities, we aim to provide a balanced view of AI, illuminating the key issues and breakthroughs. First and foremost, AI has the ability to revolutionize industries and enhance human capabilities. With the power of machine learning and deep learning algorithms, AI can process and analyze vast amounts of data at an unprecedented speed and accuracy. This has the potential to transform fields such as healthcare, finance, transportation, and manufacturing, by enabling more efficient decision-making and problem-solving. For instance, AI-powered medical imaging can assist doctors in diagnosing diseases, while self-driving cars can reduce accidents and improve tran

[543, 651, 1152, 3934, 974, 2723, 4767, 1466, 1169, 226]

...

[18.289735267034988,
 17.12442396313364,
 17.599956896551728,
 13.731630452465687,
 13.576025823965477,
 15.247027136840629,
 18.216982687348068,
 12.791823061866069,
 16.780372968349017,
 13.48240707964602]

'test'

count      10.000000
mean     1760.500000
std      1534.166241
min       226.000000
25%       731.750000
50%      1160.500000
75%      2408.750000
max      4767.000000
dtype: float64

count    10.000000
mean     17.715706
std      13.705576
min       2.874729
25%       4.889266
50%      14.919141
75%      30.519241
max      38.653522
dtype: float64

count    10.000000
mean     15.684039
std       2.155395
min      12.791823
25%      13.614927
50%      16.013700
75%      17.481074
max      18.289735
dtype: float64

In [12]:
print(pd.Series(word_counts).describe())
print(pd.Series(flesch_reading_ease).describe())
print(pd.Series(flesch_kincaid_grade).describe())

count      10.000000
mean     1760.500000
std      1534.166241
min       226.000000
25%       731.750000
50%      1160.500000
75%      2408.750000
max      4767.000000
dtype: float64
count    10.000000
mean     17.715706
std      13.705576
min       2.874729
25%       4.889266
50%      14.919141
75%      30.519241
max      38.653522
dtype: float64
count    10.000000
mean     15.684039
std       2.155395
min      12.791823
25%      13.614927
50%      16.013700
75%      17.481074
max      18.289735
dtype: float64
